# Topic 3: Reading File Formats — CSV, JSON, Parquet, ORC

> **Phase 3 → Week 3 → Topic 3**

---

## The Warehouse Analogy

Different file formats are like different ways to store goods in a warehouse:

| Format | Analogy | Characteristic |
|--------|---------|---------------|
| **CSV** | Cardboard boxes in a row | Simple, readable, but wasteful — no compression, no schema |
| **JSON** | Labelled bags with nested compartments | Flexible, self-describing, but verbose and slow to parse |
| **Parquet** | Vacuum-sealed vertical columns | Compressed, columnar — fast for analytics, gold standard |
| **ORC** | Similar to Parquet | Hive ecosystem standard, slightly better compression |

---

## Why Parquet is the Gold Standard

### Row-based (CSV) vs Columnar (Parquet)

```
CSV (row-based):
  Row 1: Alice,  Engineering, 95000, India, 2021-03-15
  Row 2: Bob,    Engineering, 88000, USA,   2020-07-01
  Row 3: Carol,  Marketing,   72000, India, 2022-01-10

  Query: SELECT AVG(salary) FROM employees
  Must read: ALL 5 columns across ALL rows (even name, dept, country not needed!)

Parquet (columnar):
  id column:      [1, 2, 3, 4, 5, ...]
  name column:    [Alice, Bob, Carol, ...]
  dept column:    [Engineering, Engineering, Marketing, ...]
  salary column:  [95000, 88000, 72000, ...]  ← read ONLY this
  country column: [India, USA, India, ...]

  Query: SELECT AVG(salary) FROM employees
  Reads: ONLY the salary column → massive I/O savings!
```

### Parquet Advantages
1. **Column pruning** — read only columns you need
2. **Predicate pushdown** — filter at the file level (skip row groups that don't match)
3. **Compression** — each column has similar data → compresses extremely well
4. **Schema embedded** — no need for separate DDL or schema files
5. **Splittable** — large files can be processed by multiple tasks in parallel

### File Format Comparison

| Feature | CSV | JSON | Parquet | ORC |
|---------|-----|------|---------|-----|
| Human readable | ✅ | ✅ | ❌ | ❌ |
| Schema embedded | ❌ | ❌ | ✅ | ✅ |
| Compression | ❌ | ❌ | ✅ (excellent) | ✅ (excellent) |
| Columnar | ❌ | ❌ | ✅ | ✅ |
| Nested data | ❌ | ✅ | ✅ | ✅ |
| Predicate pushdown | ❌ | ❌ | ✅ | ✅ |
| Splittable | ✅ (with GZIP: ❌) | ❌ | ✅ | ✅ |
| Typical size ratio | 100% | 150% | 10-25% | 8-20% |
| Best use | Data exchange, small data | APIs, semi-structured | Analytics, data lakes | Hive/ORC ecosystem |

**Production rule: Store all intermediate and final data as Parquet.**

---

## The Spark Reader API Pattern

```python
# All formats follow this pattern:
df = spark.read \
    .format("csv")                    # OR .csv(), .json(), .parquet(), .orc()
    .option("key", "value")           # format-specific options
    .schema(my_schema)                # optional — explicit schema
    .load("path/to/data")

# Shorthand:
df = spark.read.csv("path", header=True, inferSchema=True)
df = spark.read.json("path")
df = spark.read.parquet("path")
df = spark.read.orc("path")
```

---

## Interview Cheat Sheet

**Q: Why is Parquet preferred over CSV for big data?**
> Parquet is columnar — it stores each column separately. This enables column pruning (only the needed columns are read from disk, not all columns) and predicate pushdown (row groups that don't match filters are skipped entirely). It also has excellent compression because each column contains the same data type (similar values compress well). CSV reads all columns even if only one is needed, has no compression by default, and embeds no schema.

**Q: What is predicate pushdown?**
> Predicate pushdown means pushing filter conditions down to the data source level so that non-matching data is never loaded into Spark memory. With Parquet, if you filter `salary > 100000`, Spark reads each row group's min/max statistics — if a row group's max salary is 80000, the entire row group is skipped without reading it. This can reduce I/O by 90%+ on well-partitioned data.

**Q: What is the difference between single-line and multi-line JSON in Spark?**
> By default, Spark expects JSON files where each line is a separate, complete JSON object (NDJSON / JSON Lines format). If your JSON file has one large object spanning multiple lines (pretty-printed), you must use `.option("multiLine", True)`. Multi-line JSON cannot be split across tasks (the whole file must be read by one task), so avoid it for large files.

**Q: How do you read a directory of CSV files?**
> Pass the directory path: `spark.read.csv("s3://bucket/data/")`. Spark reads all files matching the format in that directory. You can also use wildcards: `spark.read.csv("s3://bucket/data/2024-01-*.csv")`. This is how partitioned data lakes are typically read.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

spark = SparkSession.builder \
    .appName("Week3 - File Formats") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark ready")

## Part 1: Reading CSV

In [ ]:
# Basic CSV read — worst practice (inferSchema)
df_bad = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/employees.csv")
print("inferSchema result — join_date comes in as String, not Date:")
df_bad.printSchema()
df_bad.show(3)

In [ ]:
# Best practice — explicit schema
emp_schema = StructType([
    StructField("id",        IntegerType(), False),
    StructField("name",      StringType(),  True),
    StructField("dept",      StringType(),  True),
    StructField("salary",    IntegerType(), True),
    StructField("country",   StringType(),  True),
    StructField("join_date", DateType(),    True),
    StructField("active",    BooleanType(), True),
])

df = spark.read \
    .schema(emp_schema) \
    .option("header",     True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .option("nullValue",  "NULL") \
    .csv("data/employees.csv")

print("Explicit schema — correct types:")
df.printSchema()
df.show()

In [ ]:
# All important CSV options
print("=== CSV READ OPTIONS REFERENCE ===")
print("""
Option                Default       Description
──────────────────────────────────────────────────────────────────
header                false         First row = column names
sep / delimiter       ,             Column separator
quote                 "             Quote character for values with delimiter
escapeChar            \\             Escape character inside quoted strings
nullValue             (empty)       String to treat as null
nanValue              NaN           String to treat as NaN
positiveInf           Inf           String for positive infinity
negativeInf           -Inf          String for negative infinity
dateFormat            yyyy-MM-dd    Date parsing format
timestampFormat       default ISO   Timestamp parsing format
multiLine             false         Rows can span multiple lines
encoding              UTF-8         File encoding
ignoreLeadingWhiteSpace false       Trim leading spaces from values
ignoreTrailingWhiteSpace false      Trim trailing spaces from values
mode                  PERMISSIVE    Error handling: PERMISSIVE / DROPMALFORMED / FAILFAST
corruptRecordColumnName _corrupt_record  Column to put bad records in PERMISSIVE mode
""")

# PERMISSIVE mode (default) — keeps bad records with nulls
df_perm = spark.read \
    .schema(emp_schema) \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .option("dateFormat", "yyyy-MM-dd") \
    .csv("data/employees.csv")
df_perm.show(3)

## Part 2: Reading JSON

In [ ]:
# JSON read — schema is inferred by default (Spark reads the JSON)
# Our products.json is NDJSON (one JSON object per line)
json_df = spark.read.json("data/products.json")
print("JSON read (auto schema):")
json_df.printSchema()
json_df.show(truncate=False)

In [ ]:
# JSON with explicit schema
product_schema = StructType([
    StructField("product_id", StringType(),  False),
    StructField("name",       StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("price",      DoubleType(),  True),
    StructField("stock",      IntegerType(), True),
])

json_df2 = spark.read \
    .schema(product_schema) \
    .json("data/products.json")
print("JSON with explicit schema:")
json_df2.printSchema()
json_df2.show()

# Write nested JSON for multiLine demo
nested_json_path = "/tmp/nested_products.json"
nested_json = """{"store": "Main", "products": [{"id": "P001", "price": 999}, {"id": "P002", "price": 29}]}
{"store": "Branch", "products": [{"id": "P003", "price": 149}]}"""
with open(nested_json_path, 'w') as f:
    f.write(nested_json)

print("\nNDJSON (each line = one JSON object) — default:")
spark.read.json(nested_json_path).show(truncate=False)

In [ ]:
# Key JSON options
print("=== JSON READ OPTIONS ===")
print("""
Option              Default       Description
──────────────────────────────────────────────────────────────────
multiLine           false         True for pretty-printed (multi-line) JSON files
dateFormat          yyyy-MM-dd    Date format string
timestampFormat     default       Timestamp format
allowComments       false         Allow Java/C++ style comments in JSON
allowUnquotedFieldNames false     Allow field names without quotes
primitivesAsString  false         Read all primitive values as strings
mode                PERMISSIVE    PERMISSIVE / DROPMALFORMED / FAILFAST
""")

## Part 3: Reading & Writing Parquet

In [ ]:
import shutil

# First: write our employee DataFrame as Parquet
parquet_path = "/tmp/employees_parquet"
if os.path.exists(parquet_path):
    shutil.rmtree(parquet_path)

df.write.mode("overwrite").parquet(parquet_path)

files = os.listdir(parquet_path)
print(f"Parquet output files: {files}")
print(f"Schema is embedded inside the .parquet files — no separate schema needed!")

In [ ]:
# Reading Parquet — NO schema needed (schema embedded in file)
df_parquet = spark.read.parquet(parquet_path)
print("Parquet read — schema auto-loaded from file:")
df_parquet.printSchema()
df_parquet.show()

# Predicate pushdown — Spark filters at the Parquet level
print("\nFilter with predicate pushdown:")
filtered = df_parquet.filter(F.col("salary") > 80000)
filtered.show()

# Check the physical plan to see 'PushedFilters'
print("\nexplain() — look for 'PushedFilters':")
filtered.explain()

In [ ]:
# Partitioned Parquet — the production pattern
partitioned_path = "/tmp/employees_partitioned"
if os.path.exists(partitioned_path):
    shutil.rmtree(partitioned_path)

# Write partitioned by country — creates subdirectory per value
df.write.mode("overwrite").partitionBy("country").parquet(partitioned_path)

print("Partitioned Parquet structure:")
for root, dirs, files_list in os.walk(partitioned_path):
    level = root.replace(partitioned_path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level == 1:  # show only parquet files in partitions
        for f in [x for x in files_list if x.endswith('.parquet')]:
            print(f'{indent}  {f}')

In [ ]:
# Partition pruning — reading only India partition
print("Read ALL partitions:")
df_all = spark.read.parquet(partitioned_path)
print(f"  Rows: {df_all.count()}")
df_all.show(3)

print("\nRead ONLY India partition (partition pruning — other folders not touched):")
df_india = spark.read.parquet(partitioned_path).filter(F.col("country") == "India")
print(f"  Rows: {df_india.count()}")
df_india.show()

# Alternatively, read specific partition directory directly
df_india_direct = spark.read.parquet(f"{partitioned_path}/country=India")
print(f"\nRead India folder directly: {df_india_direct.count()} rows")

## Part 4: Reading ORC

In [ ]:
orc_path = "/tmp/employees_orc"
if os.path.exists(orc_path):
    shutil.rmtree(orc_path)

# Write ORC
df.write.mode("overwrite").orc(orc_path)

# Read ORC — same API as Parquet
df_orc = spark.read.orc(orc_path)
print("ORC read — identical API to Parquet:")
df_orc.printSchema()
df_orc.show(3)
print()
print("Parquet vs ORC in practice:")
print("  Parquet: standard for most modern data lakes (AWS, GCP, Azure, Databricks)")
print("  ORC: preferred in Hive/Hadoop ecosystem, slightly better compression")
print("  Both support predicate pushdown and column pruning")
print("  Production rule: use Parquet unless working with Hive tables")

## Part 5: Writing Data — Modes and Options

In [ ]:
print("=== WRITE MODES ===")
print("""
Mode         Behaviour if path exists
──────────────────────────────────────────────────────────────────
overwrite    Delete existing data and write fresh  ← most common in ETL
append       Add new data to existing data
error        Raise an error (default)              ← safe default
ignore       Do nothing if data already exists
""")

out_path = "/tmp/write_demo"
if os.path.exists(out_path):
    shutil.rmtree(out_path)

# Write Parquet with options
df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(out_path)
print(f"Written to: {out_path}")
print(f"Files: {os.listdir(out_path)}")

print("""
Compression options for Parquet:
  snappy   → fast read/write, moderate compression (DEFAULT)
  gzip     → better compression, slower read/write
  lz4      → fastest, least compression
  zstd     → best balance (Spark 3.x, recommended)
  none     → no compression
""")

In [ ]:
# Write CSV with options
csv_out = "/tmp/employees_out.csv"
if os.path.exists(csv_out):
    shutil.rmtree(csv_out)

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .option("sep", ",") \
    .option("dateFormat", "yyyy-MM-dd") \
    .csv(csv_out)

print("CSV output files (one per partition):")
for f in os.listdir(csv_out):
    print(f"  {f}")

# Coalesce to 1 file (common for small output)
one_file_path = "/tmp/employees_one.csv"
if os.path.exists(one_file_path):
    shutil.rmtree(one_file_path)

df.coalesce(1).write.mode("overwrite").option("header", True).csv(one_file_path)
print("\nAfter coalesce(1) — single output file:")
for f in [x for x in os.listdir(one_file_path) if x.endswith('.csv')]:
    print(f"  {f}")
print("Note: coalesce to 1 causes all data to flow to one executor — only for small output!")

## Exercises

1. Write the products JSON DataFrame as Parquet. Then read it back and run `explain()` on a filter query. Can you see `PushedFilters` in the output?
2. Write employees DataFrame partitioned by both `dept` AND `country`. List all the subdirectory combinations created.
3. Read the CSV with `mode=FAILFAST`. Then introduce a bad row in the file (e.g., a string in the salary column). What happens?
4. Write to CSV with a `|` delimiter instead of comma. Then read it back using `sep="|"`.

In [ ]:
# Exercise 1: JSON → Parquet → predicate pushdown
json_to_parquet = "/tmp/products_parquet"
if os.path.exists(json_to_parquet):
    shutil.rmtree(json_to_parquet)

json_df2.write.mode("overwrite").parquet(json_to_parquet)
df_back = spark.read.parquet(json_to_parquet)

print("Exercise 1 — Parquet predicate pushdown (look for PushedFilters):")
df_back.filter(F.col("price") > 100).explain()

In [ ]:
# Exercise 2: Partition by dept AND country
multi_part = "/tmp/employees_multi_part"
if os.path.exists(multi_part):
    shutil.rmtree(multi_part)

df.write.mode("overwrite").partitionBy("dept", "country").parquet(multi_part)

print("Exercise 2 — Multi-level partitions:")
for root, dirs, files_list in os.walk(multi_part):
    level = root.replace(multi_part, '').count(os.sep)
    if level <= 2:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')